# Derf — Dynamic erf

源码导航：[core/norm/derf.py](../../../core/norm/derf.py) 中的 `Derf`。

Chen et al. (2025) 在 *Stronger Normalization-Free Transformers* 中提出 **Derf（Dynamic erf）**，在 DyT 的基础上将双曲正切替换为误差函数，并引入可学习偏移量 $s$。实验表明，Derf 在视觉（ViT）、生成（DiT）、DNA 建模（Caduceus）与语音（wav2vec 2.0）等多个领域的泛化性能均优于 DyT 与标准 LayerNorm，代表了当前非线性归一化替代方案的最前沿。

### 1. 理论推导

Derf 的核心公式为：

$$\text{Derf}(x) = \text{erf}(\alpha \cdot x + s) \odot \gamma$$

其中：
- $\alpha > 0$ 为可学习标量，控制函数的陡峭程度；
- $s \in \mathbb{R}$ 为**可学习偏移**，允许输入分布整体平移，提供额外的自由度；
- $\gamma \in \mathbb{R}^d$ 为逐通道可学习缩放（可选）。

**erf 与 tanh 的对比：**

| 属性 | tanh | erf |
|---|---|---|
| 输出范围 | $(-1, 1)$ | $(-1, 1)$ |
| 边界梯度衰减 | 较快（$\sim e^{-2x}$） | 较慢（$\sim e^{-x^2}/x$） |
| 原点线性近似 | $\tanh(x) \approx x$ | $\text{erf}(x) \approx \frac{2}{\sqrt{\pi}} x$ |
| 饱和柔和度 | 较硬 | 更柔和 |

erf 在饱和区的梯度衰减更缓慢，使得深层网络中信号传播更稳定。此外，偏移参数 $s$ 使 Derf 能够自适应地调整非对称输入的响应中心，而 DyT 的 $\tanh(\alpha x)$ 始终关于原点对称。

**训练建议**：
- 默认 $\alpha = 0.3$，比 DyT 的 $0.5$ 更保守，有助于维持近线性区。
- 搭配 Muon 优化器时，$\alpha$ 过大可能导致谱范数增长过快，建议保持 $\alpha \leq 0.3$。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.norm.derf import Derf

### 2. 形状与 dtype 检查

In [ ]:
torch.manual_seed(0)
x = torch.randn(2, 4, 16) * 3
norm = Derf(16)
y = norm(x)

print("x.shape =", tuple(x.shape))
print("y.shape =", tuple(y.shape))
print("Derf α 初始值:", norm.alpha.item())
print("Derf s 初始值:", norm.shift.item())
assert x.shape == y.shape, "Derf 必须保持输入输出维度一致！"

### 3. 饱和边界验证

erf 的输出范围天然限制在 $(-1, 1)$，因此 Derf 同样具备有界性。

In [ ]:
norm = Derf(4, use_gamma=True)
with torch.no_grad():
    norm.gamma.fill_(3.0)

x_extreme = torch.randn(10, 4) * 100.0
y = norm(x_extreme)

print("输入最大值:", x_extreme.abs().max().item())
print("输出最大值:", y.abs().max().item())
assert y.abs().max().item() <= 3.0 + 1e-5, "输出应被限制在 [-γ, γ] 内！"

### 4. 可学习偏移 $s$ 的效应

与 DyT 不同，Derf 的偏移参数 $s$ 使响应中心可移动。对同一输入，不同的 $s$ 会产生截然不同的输出分布。

In [ ]:
x = torch.randn(3, 8)

norm_s0 = Derf(8, shift_init=0.0)
norm_s2 = Derf(8, shift_init=2.0)

y_s0 = norm_s0(x)
y_s2 = norm_s2(x)

print("s=0.0 时输出均值:", y_s0.mean().item())
print("s=2.0 时输出均值:", y_s2.mean().item())
assert not torch.allclose(y_s0, y_s2, atol=1e-3), "不同 s 应产生不同输出！"

### 5. erf 与 tanh 的函数形态对比

In [ ]:
import math

z = torch.linspace(-4, 4, 200)
tanh_y = torch.tanh(z)
erf_y = torch.erf(z)

print("z 范围:", z[0].item(), "~", z[-1].item())
print("tanh(4) =", tanh_y[-1].item())
print("erf(4)  =", erf_y[-1].item())
print("两者在 z=4 时的差距:", (tanh_y[-1] - erf_y[-1]).abs().item())

# 验证 erf 的饱和更柔和：在 z=2 时，tanh 已接近 0.96，而 erf 约为 0.995
idx_2 = (z - 2.0).abs().argmin()
print(f"tanh({z[idx_2].item():.2f}) = {tanh_y[idx_2].item():.4f}")
print(f"erf({z[idx_2].item():.2f})  = {erf_y[idx_2].item():.4f}")

### 6. 源码精讲

以下为 `core/norm/derf.py` 的完整实现：

```python
class Derf(nn.Module):
    def __init__(self, normalized_shape: int, alpha_init: float = 0.3,
                 shift_init: float = 0.0, gamma_init: float = 1.0,
                 use_gamma: bool = True):
        super().__init__()
        # 对数参数化保证 α > 0
        self.log_alpha = nn.Parameter(torch.tensor(math.log(alpha_init),
                                                    dtype=torch.float32))
        # 可学习偏移 s，提供额外的分布平移自由度
        self.shift = nn.Parameter(torch.tensor(shift_init, dtype=torch.float32))
        if use_gamma:
            self.gamma = nn.Parameter(
                torch.full((normalized_shape,), gamma_init, dtype=torch.float32))
        else:
            self.register_parameter("gamma", None)

    @property
    def alpha(self) -> torch.Tensor:
        return self.log_alpha.exp()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        orig_dtype = x.dtype
        x_fp32 = x.float()
        alpha = self.alpha.to(x_fp32.dtype)
        shift = self.shift.to(x_fp32.dtype)
        # 核心：erf(αx + s)，比 tanh 多了偏移且饱和更柔和
        out = torch.erf(alpha * x_fp32 + shift)
        if self.gamma is not None:
            out = out * self.gamma.to(x_fp32.dtype)
        return out.to(orig_dtype)
```

关键设计点：
- `shift` 参数使 Derf 能够处理非对称分布的输入，而 DyT 的 $\tanh(\alpha x)$ 始终关于原点对称。
- `alpha_init=0.3` 默认比 DyT 更保守，降低深层网络饱和风险。
- `torch.erf` 在 PyTorch 2.x 中已原生支持，无需额外依赖。
- 与 DyT 相同，所有运算均为逐元素，无 reduction 开销。

---

## 延伸阅读与参考资料

### 核心论文
- **Stronger Normalization-Free Transformers**: Chen et al., 2025. [arXiv:2512.10938](https://arxiv.org/abs/2512.10938)

### 相关讨论
- **Transformers without Normalization**: Zhu et al., 2025. [arXiv:2503.10622](https://arxiv.org/abs/2503.10622) — DyT 基础工作
- **Bounded Hyperbolic Tangent**: Byun et al., 2025. [arXiv:2601.09719](https://arxiv.org/abs/2601.09719) — 另一 DyT 改进方向
- **Does Your Optimizer Care How You Normalize?**: 2025. [arXiv:2604.01563](https://arxiv.org/abs/2604.01563) — 优化器与归一化方案的耦合分析